<a href="https://colab.research.google.com/github/Jobarcellos/Artigo-M-todos-Quantitativos-Aplicados/blob/main/Artigo_M%C3%A9todos_Quantitativos_Aplicados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Determinantes da Regularidade do Corpo Docente nos Municípios Brasileiro

**Autores:** Flávio Dias Cabral, Joelma Barcellos Santanna e Kátia Soars dos Santos   

**Disciplina:** Métodos Quantitativos Aplicados  
**Professora:** Prof.ª Dr.ª Nadia Cardoso Moreira  
**Objetivo:** Reproduzir a construção da base, estatísticas descritivas, modelos em painel, diagnósticos e robustez do artigo.  
**Período analisado:** 2013–2024  
**Unidade de análise:** município-ano  
**Variável dependente:** IRD

Este notebook reproduz as etapas de validação da base, estatísticas descritivas, matriz de correlação, diagnóstico de multicolinearidade, estimação de modelos em painel, teste de Hausman e análises de robustez utilizadas no artigo.

**Unidade de análise:** município-ano  
**Período do modelo principal:** 2013 a 2024  
**Número esperado de municípios:** 5.570  
**Número esperado de observações:** 66.840  
**Variável dependente:** IRD  
**Variáveis explicativas:** ICG, AFD, IED e ATU  

## Etapa 1 — Instalação e importação dos pacotes

Nesta etapa, são instaladas e importadas as bibliotecas necessárias para leitura da base, tratamento dos dados, estimação dos modelos em painel e exportação dos resultados.

In [ ]:
# ============================================================
# 1. INSTALAÇÃO E IMPORTAÇÃO DOS PACOTES
# ============================================================

!pip -q install linearmodels

import os
import numpy as np
import pandas as pd
import statsmodels.api as sm

from scipy import stats
from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS
from statsmodels.stats.outliers_influence import variance_inflation_factor

from google.colab import drive

## Etapa 2 — Configuração dos caminhos e parâmetros gerais

Nesta etapa, são definidos os caminhos da pasta no Google Drive, o período de análise e as variáveis do modelo principal.

O modelo principal utiliza os anos de 2013 a 2024 e as variáveis IRD, ICG, AFD, IED e ATU.

In [ ]:
# ============================================================
# ETAPA 2 — CONFIGURAÇÃO REPRODUTÍVEL DOS CAMINHOS
# ============================================================

URL_BASE = "https://raw.githubusercontent.com/Jobarcellos/Artigo-M-todos-Quantitativos-Aplicados/main/dados/painel_ird_municipios_2013_2024.csv"

PASTA_SAIDA = "/content/saidas/"
os.makedirs(PASTA_SAIDA, exist_ok=True)

ANOS_MODELO = list(range(2013, 2025))  # 2013 a 2024

VARIAVEIS_MODELO = ["IRD", "ICG", "AFD", "IED", "ATU"]
PREDITORES = ["ICG", "AFD", "IED", "ATU"]

print("Base pública:", URL_BASE)
print("Pasta de saída:", PASTA_SAIDA)

Base pública: https://raw.githubusercontent.com/Jobarcellos/Artigo-M-todos-Quantitativos-Aplicados/main/dados/painel_ird_municipios_2013_2024.csv
Pasta de saída: /content/saidas/


## Etapa 3 — Leitura da base e filtro do período do estudo

Nesta etapa, é carregada a base consolidada `painel_ird_municipios.csv`.

Embora a base contenha informações até 2025, o modelo principal do artigo considera apenas o período de 2013 a 2024. Portanto, após a leitura da base, é aplicado um filtro temporal para manter somente os anos utilizados na análise.

Cada linha da base filtrada representa um município em determinado ano, compondo a estrutura de painel município-ano utilizada no estudo.

In [ ]:
# ============================================================
# ETAPA 3 — LEITURA DA BASE E FILTRO DO PERÍODO DO ESTUDO
# ============================================================

painel = pd.read_csv(URL_BASE)

painel["CO_MUNICIPIO"] = painel["CO_MUNICIPIO"].astype(str).str.strip().str.zfill(7)
painel["ANO"] = painel["ANO"].astype(int)

print("Dimensão da base original:", painel.shape)
print("Anos disponíveis na base:", [int(ano) for ano in sorted(painel["ANO"].unique())])

# O estudo considera apenas o período de 2013 a 2024
painel_modelo = painel[painel["ANO"].between(2013, 2024)].copy()

print("\nDimensão da base utilizada no modelo principal:", painel_modelo.shape)
print("Anos considerados no estudo:", [int(ano) for ano in sorted(painel_modelo["ANO"].unique())])
print("Número de municípios:", painel_modelo["CO_MUNICIPIO"].nunique())

painel_modelo.head()

Dimensão da base original: (66840, 10)
Anos disponíveis na base: [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

Dimensão da base utilizada no modelo principal: (66840, 10)
Anos considerados no estudo: [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Número de municípios: 5570


,CO_MUNICIPIO,IRD,ANO,ICG,AFD,IED,ATU,HAD,N_ESCOLAS,LN_ESC
0,1100015,3.646240,2013,1.869565,20.613953,65.072093,12.425581,NaN,43.0,3.761200
1,1100023,3.199682,2013,3.842105,60.612903,24.496774,25.000000,NaN,35.0,3.555348
2,1100031,3.771412,2013,2.375000,65.766667,49.250000,13.333333,NaN,8.0,2.079442
3,1100049,2.917221,2013,2.795918,43.233333,24.407692,22.669231,NaN,47.0,3.850148
4,1100056,2.614694,2013,2.533333,71.118182,45.154545,19.872727,NaN,15.0,2.708050


## Etapa 4 — Filtro do período analisado

A base original contém informações de 2013 a 2025. No entanto, o modelo principal do artigo utiliza apenas o período de 2013 a 2024.

Por isso, nesta etapa, a base é filtrada para manter somente os anos utilizados na estimação principal.

In [ ]:
# ============================================================
# ETAPA 4 — FILTRO DO PERÍODO ANALISADO
# ============================================================

painel_modelo = painel[painel["ANO"].isin(ANOS_MODELO)].copy()

print("Dimensão da base filtrada:", painel_modelo.shape)
print("Anos do modelo:", [int(ano) for ano in sorted(painel_modelo["ANO"].unique())])
print("Número de municípios:", painel_modelo["CO_MUNICIPIO"].nunique())

Dimensão da base filtrada: (66840, 10)
Anos do modelo: [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Número de municípios: 5570


## Etapa 5 — Conferência inicial da estrutura do painel

Após o filtro temporal, é realizada uma conferência inicial da estrutura da base.

Para o modelo principal, espera-se um painel balanceado com:

5.570 municípios × 12 anos = 66.840 observações.

In [ ]:
# ============================================================
# ETAPA 5 — CONFERÊNCIA INICIAL DO PAINEL
# ============================================================

n_obs = painel_modelo.shape[0]
n_municipios = painel_modelo["CO_MUNICIPIO"].nunique()
n_anos = painel_modelo["ANO"].nunique()

print("Observações:", n_obs)
print("Municípios:", n_municipios)
print("Anos:", n_anos)
print("Municípios x anos:", n_municipios * n_anos)

Observações: 66840
Municípios: 5570
Anos: 12
Municípios x anos: 66840


## Etapa 6 — Conferência de municípios por ano

Nesta etapa, verifica-se se todos os anos do período analisado possuem 5.570 municípios.

Essa conferência confirma se o painel está balanceado ao longo do tempo.

In [ ]:
# ============================================================
# ETAPA 6 — CONFERÊNCIA DE MUNICÍPIOS POR ANO
# ============================================================

municipios_por_ano = painel_modelo.groupby("ANO")["CO_MUNICIPIO"].nunique()

print(municipios_por_ano)

ANO
2013    5570
2014    5570
2015    5570
2016    5570
2017    5570
2018    5570
2019    5570
2020    5570
2021    5570
2022    5570
2023    5570
2024    5570
Name: CO_MUNICIPIO, dtype: int64


## Etapa 7 — Verificação de duplicidades município-ano

Nesta etapa, verifica-se se existe duplicidade na combinação município-ano.

Em um painel corretamente estruturado, cada município deve aparecer apenas uma vez em cada ano.

In [ ]:
# ============================================================
# ETAPA 7 — VERIFICAÇÃO DE DUPLICIDADES MUNICÍPIO-ANO
# ============================================================

duplicados = painel_modelo[
    painel_modelo.duplicated(subset=["CO_MUNICIPIO", "ANO"], keep=False)
].sort_values(["CO_MUNICIPIO", "ANO"])

print("Quantidade de linhas duplicadas:", duplicados.shape[0])

duplicados.head(20)

Quantidade de linhas duplicadas: 0


,CO_MUNICIPIO,IRD,ANO,ICG,AFD,IED,ATU,HAD,N_ESCOLAS,LN_ESC


## Etapa 8 — Conferência de dados faltantes nas variáveis principais

Nesta etapa, verifica-se a existência de dados faltantes nas variáveis utilizadas no modelo principal: IRD, ICG, AFD, IED e ATU.

A estimação principal depende de observações completas para essas variáveis.

In [ ]:
# ============================================================
# ETAPA 8 — CONFERÊNCIA DE DADOS FALTANTES
# ============================================================

faltantes = painel_modelo[VARIAVEIS_MODELO].isna().sum()

print(faltantes)

IRD    0
ICG    0
AFD    0
IED    0
ATU    0
dtype: int64


## Etapa 9 — Validação final da base do modelo principal

Nesta etapa, são aplicados testes automáticos para confirmar que o painel principal contém exatamente 66.840 observações, 5.570 municípios, 12 anos e nenhuma ausência nas variáveis centrais do modelo.

In [ ]:
# ============================================================
# ETAPA 9 — VALIDAÇÃO FINAL DA BASE
# ============================================================

assert painel_modelo.shape[0] == 66840, "Erro: número de observações diferente de 66.840."
assert painel_modelo["CO_MUNICIPIO"].nunique() == 5570, "Erro: número de municípios diferente de 5.570."
assert painel_modelo["ANO"].nunique() == 12, "Erro: número de anos diferente de 12."
assert painel_modelo[VARIAVEIS_MODELO].isna().sum().sum() == 0, "Erro: há dados faltantes nas variáveis principais."
assert duplicados.shape[0] == 0, "Erro: há duplicidades município-ano."

print("Painel principal validado com sucesso.")

Painel principal validado com sucesso.


## Etapa 10 — Estatísticas descritivas

Nesta etapa, são calculadas as estatísticas descritivas das variáveis do modelo.

São apresentados média, mediana, desvio-padrão, valor mínimo e valor máximo. Esses resultados correspondem à caracterização inicial da base utilizada no artigo.

In [ ]:
# ============================================================
# ETAPA 10 — ESTATÍSTICAS DESCRITIVAS
# ============================================================

desc = painel_modelo[VARIAVEIS_MODELO].describe().T
desc = desc[["mean", "50%", "std", "min", "max"]]
desc.columns = ["Média", "Mediana", "DP", "Mín", "Máx"]

desc = desc.round(3)

desc

,Média,Mediana,DP,Mín,Máx
IRD,3.073,3.075,0.442,1.388,4.804
ICG,2.582,2.500,0.560,1.111,5.500
AFD,52.834,55.317,22.182,0.092,100.000
IED,32.293,29.200,18.601,0.000,93.443
ATU,18.848,18.750,4.322,4.284,39.957


In [ ]:
# Exportação da tabela de estatísticas descritivas

desc.to_excel(PASTA_SAIDA + "Tabela_2_estatisticas_descritivas.xlsx")

print("Tabela de estatísticas descritivas exportada.")

Tabela de estatísticas descritivas exportada.


## Etapa 10.1 — Tabela 3 — IRD médio municipal por região (2013–2024)
Nesta etapa, calcula-se o IRD médio dos municípios brasileiros por região, considerando o período de 2013 a 2024.

A tabela apresenta a média regional do IRD, o desvio-padrão e o número de municípios em cada região. Essa análise permite observar diferenças regionais na regularidade docente antes da estimação dos modelos econométricos.

In [ ]:
# ============================================================
# ETAPA 10.1 — IRD MÉDIO MUNICIPAL POR REGIÃO (2013–2024)
# ============================================================

# Cria uma cópia da base do modelo principal
df_regiao = painel_modelo.copy()

# Garante que o código do município esteja em formato texto com 7 dígitos
df_regiao["CO_MUNICIPIO"] = df_regiao["CO_MUNICIPIO"].astype(str).str.zfill(7)

# ------------------------------------------------------------
# Criação da variável REGIAO, caso ela ainda não exista na base
# ------------------------------------------------------------

if "REGIAO" not in df_regiao.columns:

    mapa_regioes = {
        "1": "Norte",
        "2": "Nordeste",
        "3": "Sudeste",
        "4": "Sul",
        "5": "Centro-Oeste"
    }

    df_regiao["REGIAO"] = (
        df_regiao["CO_MUNICIPIO"]
        .str[0]
        .map(mapa_regioes)
    )

# Conferência de possíveis códigos sem região
sem_regiao = df_regiao["REGIAO"].isna().sum()

print("Observações sem identificação de região:", sem_regiao)

# ------------------------------------------------------------
# Cálculo do IRD médio por região
# ------------------------------------------------------------

tabela_ird_regiao = (
    df_regiao
    .groupby("REGIAO")
    .agg(
        IRD_medio=("IRD", "mean"),
        Desvio_padrao=("IRD", "std"),
        Municipios=("CO_MUNICIPIO", "nunique")
    )
    .reset_index()
)

# Ordena da maior para a menor média de IRD
tabela_ird_regiao = tabela_ird_regiao.sort_values(
    by="IRD_medio",
    ascending=False
)

# Arredondamento para apresentação
tabela_ird_regiao_apresentacao = tabela_ird_regiao.copy()
tabela_ird_regiao_apresentacao["IRD_medio"] = tabela_ird_regiao_apresentacao["IRD_medio"].round(3)
tabela_ird_regiao_apresentacao["Desvio_padrao"] = tabela_ird_regiao_apresentacao["Desvio_padrao"].round(3)

# Exibe a tabela
tabela_ird_regiao_apresentacao

Observações sem identificação de região: 0


,REGIAO,IRD_medio,Desvio_padrao,Municipios
1,Nordeste,3.184,0.465,1794
0,Centro-Oeste,3.049,0.367,467
4,Sul,3.034,0.459,1191
2,Norte,3.030,0.441,450
3,Sudeste,3.001,0.397,1668


## Etapa 11 — Matriz de correlação

Nesta etapa, é calculada a matriz de correlação entre as variáveis do modelo.

A matriz permite observar o grau de associação linear entre os indicadores. Essa análise é preliminar e não permite inferir causalidade.

In [ ]:
# ============================================================
# ETAPA 11 — MATRIZ DE CORRELAÇÃO
# ============================================================

corr = painel_modelo[VARIAVEIS_MODELO].corr().round(3)

corr

,IRD,ICG,AFD,IED,ATU
IRD,1.000,0.097,-0.026,0.107,-0.113
ICG,0.097,1.000,-0.016,-0.146,0.216
AFD,-0.026,-0.016,1.000,-0.484,0.402
IED,0.107,-0.146,-0.484,1.000,-0.362
ATU,-0.113,0.216,0.402,-0.362,1.000


In [ ]:
# Exportação da matriz de correlação

corr.to_excel(PASTA_SAIDA + "Tabela_4_matriz_correlacao.xlsx")

print("Matriz de correlação exportada.")

Matriz de correlação exportada.


## Etapa 12 — Diagnóstico de multicolinearidade transversal

Nesta etapa, é calculado o Fator de Inflação da Variância (VIF) com os dados em nível transversal agrupado.

Esse diagnóstico permite verificar se há forte associação entre os preditores quando se considera a base completa sem retirar os efeitos fixos dos municípios.

In [ ]:
# ============================================================
# ETAPA 12 — VIF TRANSVERSAL
# ============================================================

X_vif = painel_modelo[PREDITORES].dropna().copy()
X_vif_const = sm.add_constant(X_vif)

vif_transversal = pd.DataFrame()
vif_transversal["Variável"] = X_vif_const.columns
vif_transversal["VIF"] = [
    variance_inflation_factor(X_vif_const.values, i)
    for i in range(X_vif_const.shape[1])
]

vif_transversal = vif_transversal.round(3)

vif_transversal

,Variável,VIF
0,const,57.691
1,ICG,1.082
2,AFD,1.452
3,IED,1.390
4,ATU,1.307


## Etapa 13 — Diagnóstico de multicolinearidade intragrupo

Como o modelo principal utiliza Efeitos Fixos, a variação mais relevante é a variação dentro de cada município ao longo do tempo.

Por isso, nesta etapa, o VIF é recalculado após retirar a média municipal de cada variável, permitindo avaliar a multicolinearidade na dimensão efetivamente utilizada pelo modelo de Efeitos Fixos.

In [ ]:
# ============================================================
# ETAPA 13 — VIF INTRAGRUPO
# ============================================================

df_within = painel_modelo[["CO_MUNICIPIO"] + PREDITORES].copy()

for var in PREDITORES:
    df_within[var + "_within"] = df_within[var] - df_within.groupby("CO_MUNICIPIO")[var].transform("mean")

within_vars = [var + "_within" for var in PREDITORES]

X_vif_within = df_within[within_vars].dropna().copy()
X_vif_within_const = sm.add_constant(X_vif_within)

vif_within = pd.DataFrame()
vif_within["Variável"] = X_vif_within_const.columns
vif_within["VIF"] = [
    variance_inflation_factor(X_vif_within_const.values, i)
    for i in range(X_vif_within_const.shape[1])
]

vif_within = vif_within.round(3)

vif_within

,Variável,VIF
0,const,1.000
1,ICG_within,1.015
2,AFD_within,1.035
3,IED_within,1.084
4,ATU_within,1.047


In [ ]:
# Exportação dos VIFs

with pd.ExcelWriter(PASTA_SAIDA + "Tabela_VIF.xlsx") as writer:
    vif_transversal.to_excel(writer, sheet_name="VIF_transversal", index=False)
    vif_within.to_excel(writer, sheet_name="VIF_intragrupo", index=False)

print("Tabelas de VIF exportadas.")

Tabelas de VIF exportadas.


## Etapa 14 — Preparação da base para modelos em painel

Nesta etapa, a base é organizada no formato necessário para estimação dos modelos em painel.

O índice é definido pela combinação entre município e ano, pois o estudo acompanha os mesmos municípios ao longo do tempo. A variável dependente é o IRD, e os preditores do modelo principal são ICG, AFD, IED e ATU.

In [ ]:
# ============================================================
# ETAPA 14 — PREPARAÇÃO DA BASE PARA MODELOS EM PAINEL
# ============================================================

from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS

# Define os preditores do modelo principal
PREDITORES = ["ICG", "AFD", "IED", "ATU"]

# Mantém apenas observações completas do modelo principal
df_modelos = (
    painel_modelo[["CO_MUNICIPIO", "ANO", "IRD"] + PREDITORES]
    .dropna()
    .copy()
)

# Organiza a base em formato painel
df_panel = (
    df_modelos
    .set_index(["CO_MUNICIPIO", "ANO"])
    .sort_index()
)

# Variável dependente
y = df_panel["IRD"]

# Variáveis explicativas com constante
X = sm.add_constant(df_panel[PREDITORES])

print("Dimensão da base usada nos modelos:", df_panel.shape)
print("Número de municípios:", df_panel.index.get_level_values("CO_MUNICIPIO").nunique())
print("Número de anos:", df_panel.index.get_level_values("ANO").nunique())

df_panel.head()

Dimensão da base usada nos modelos: (66840, 5)
Número de municípios: 5570
Número de anos: 12


IRD       ICG        AFD        IED        ATU
CO_MUNICIPIO ANO                                                      
1100015      2013  3.646240  1.869565  20.613953  65.072093  12.425581
             2014  3.582580  2.171429  31.331250  58.465625  14.825000
             2015  3.603987  1.918919  30.555882  63.905882  14.676471
             2016  3.137994  1.944444  29.957576  65.027273  14.472727
             2017  2.964935  2.000000  30.716129  59.203226  14.980645

## Etapa 14 — Preparação da base para modelos em painel

Nesta etapa, a base é organizada no formato adequado para estimação dos modelos em painel.

O índice é definido pela combinação entre município e ano, pois a análise acompanha os mesmos municípios ao longo do tempo.

In [ ]:
# ============================================================
# ETAPA 14 — PREPARAÇÃO DA BASE PARA MODELOS EM PAINEL
# ============================================================

df_panel = painel_modelo.set_index(["CO_MUNICIPIO", "ANO"]).sort_index()

y = df_panel["IRD"]
X = df_panel[PREDITORES]
X = sm.add_constant(X)

print("Dimensão de y:", y.shape)
print("Dimensão de X:", X.shape)

df_panel.head()

Dimensão de y: (66840,)
Dimensão de X: (66840, 5)


IRD       ICG        AFD        IED        ATU  \
CO_MUNICIPIO ANO                                                         
1100015      2013  3.646240  1.869565  20.613953  65.072093  12.425581   
             2014  3.582580  2.171429  31.331250  58.465625  14.825000   
             2015  3.603987  1.918919  30.555882  63.905882  14.676471   
             2016  3.137994  1.944444  29.957576  65.027273  14.472727   
             2017  2.964935  2.000000  30.716129  59.203226  14.980645   

                        HAD  N_ESCOLAS    LN_ESC  
CO_MUNICIPIO ANO                                  
1100015      2013       NaN       43.0  3.761200  
             2014       NaN       34.0  3.526361  
             2015       NaN       34.0  3.526361  
             2016  4.000000       33.0  3.496508  
             2017  4.005882       32.0  3.465736

## Etapa 15 — Estimação do modelo Pooled OLS

Nesta etapa, é estimado o modelo de Mínimos Quadrados Agrupados.

Esse modelo considera todas as observações de forma conjunta, sem controlar as características permanentes de cada município. Ele é utilizado como especificação inicial de comparação.

In [ ]:
# ============================================================
# ETAPA 15 — MODELO POOLED OLS
# ============================================================

modelo_pooled = PooledOLS(y, X)
res_pooled = modelo_pooled.fit(cov_type="clustered", cluster_entity=True)

print(res_pooled.summary)

                          PooledOLS Estimation Summary                          
Dep. Variable:                    IRD   R-squared:                        0.0403
Estimator:                  PooledOLS   R-squared (Between):              0.0547
No. Observations:               66840   R-squared (Within):               0.0106
Date:                Sun, Jul 05 2026   R-squared (Overall):              0.0403
Time:                        03:15:37   Log-likelihood                -3.883e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      702.02
Entities:                        5570   P-value                           0.0000
Avg Obs:                       12.000   Distribution:                 F(4,66835)
Min Obs:                       12.000                                           
Max Obs:                       12.000   F-statistic (robust):             110.19
                            

## Etapa 16 — Estimação do modelo de Efeitos Fixos

Nesta etapa, é estimado o modelo de Efeitos Fixos com efeitos de município e de ano.

Esse modelo controla características não observadas e constantes ao longo do tempo em cada município, além de choques comuns a todos os municípios em determinado ano.

In [ ]:
# ============================================================
# ETAPA 16 — MODELO DE EFEITOS FIXOS
# ============================================================

modelo_fe = PanelOLS(
    y,
    X,
    entity_effects=True,
    time_effects=True
)

res_fe = modelo_fe.fit(cov_type="clustered", cluster_entity=True)

print(res_fe.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                    IRD   R-squared:                        0.0112
Estimator:                   PanelOLS   R-squared (Between):              0.0446
No. Observations:               66840   R-squared (Within):               0.0089
Date:                Sun, Jul 05 2026   R-squared (Overall):              0.0329
Time:                        03:15:40   Log-likelihood                    494.06
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      174.12
Entities:                        5570   P-value                           0.0000
Avg Obs:                       12.000   Distribution:                 F(4,61255)
Min Obs:                       12.000                                           
Max Obs:                       12.000   F-statistic (robust):             47.042
                            

## Etapa 17 — Cálculo do R² inclusivo

Nesta etapa, é calculado o R² inclusivo do modelo de Efeitos Fixos.

O R² within mostra a capacidade dos preditores de explicar a variação do IRD dentro dos municípios ao longo do tempo. Já o R² inclusivo considera o modelo completo, incluindo os efeitos fixos de município e de ano.

In [ ]:
# ============================================================
# ETAPA 17 — CÁLCULO DO R² INCLUSIVO
# ============================================================

r2_incl = 1 - np.var(np.asarray(res_fe.resids)) / np.var(np.asarray(y))

print("=== R² DO MODELO DE EFEITOS FIXOS ===")
print(f"R² inclusivo = {r2_incl:.4f}")
print(f"R² within    = {res_fe.rsquared:.4f}")

=== R² DO MODELO DE EFEITOS FIXOS ===
R² inclusivo = 0.7041
R² within    = 0.0112


In [ ]:
# ============================================================
# CONFERÊNCIA E RECONSTRUÇÃO DOS OBJETOS DOS MODELOS
# ============================================================

from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS

# Garante os preditores do modelo principal
PREDITORES = ["ICG", "AFD", "IED", "ATU"]

# Recria a base em formato painel
df_panel = (
    painel_modelo[["CO_MUNICIPIO", "ANO", "IRD"] + PREDITORES]
    .dropna()
    .copy()
    .set_index(["CO_MUNICIPIO", "ANO"])
    .sort_index()
)

# Variável dependente e matriz de preditores
y = df_panel["IRD"]
X = sm.add_constant(df_panel[PREDITORES])

print("Base reconstruída com sucesso.")
print("Dimensão:", df_panel.shape)
print("Municípios:", df_panel.index.get_level_values("CO_MUNICIPIO").nunique())
print("Anos:", df_panel.index.get_level_values("ANO").nunique())

Base reconstruída com sucesso.
Dimensão: (66840, 5)
Municípios: 5570
Anos: 12


## Etapa 18 — Estimação do modelo de Efeitos Aleatórios

Nesta etapa, é estimado o modelo de Efeitos Aleatórios.

Esse modelo assume que as diferenças não observadas entre municípios não estão correlacionadas com as variáveis explicativas. Esse pressuposto será avaliado por meio do teste de Hausman.

In [ ]:
# ============================================================
# ETAPA 18 — MODELO DE EFEITOS ALEATÓRIOS
# ============================================================

modelo_re = RandomEffects(y, X)
res_re = modelo_re.fit(cov_type="clustered", cluster_entity=True)

print(res_re.summary)

                        RandomEffects Estimation Summary                        
Dep. Variable:                    IRD   R-squared:                        0.0259
Estimator:              RandomEffects   R-squared (Between):              0.0190
No. Observations:               66840   R-squared (Within):               0.0266
Date:                Sun, Jul 05 2026   R-squared (Overall):              0.0214
Time:                        03:18:42   Log-likelihood                   -4817.9
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      444.76
Entities:                        5570   P-value                           0.0000
Avg Obs:                       12.000   Distribution:                 F(4,66835)
Min Obs:                       12.000                                           
Max Obs:                       12.000   F-statistic (robust):             132.41
                            

In [ ]:
# ============================================================
# TABELA-RESUMO DO AJUSTE DOS MODELOS COM R² INCLUSIVO
# ============================================================

# Recalcula o R² inclusivo do modelo de Efeitos Fixos
residuos_fe = np.asarray(res_fe.resids)

# Usa a mesma variável dependente do modelo de efeitos fixos
y_fe = np.asarray(y.loc[res_fe.resids.index])

rss_fe = np.sum(residuos_fe ** 2)
tss_fe = np.sum((y_fe - y_fe.mean()) ** 2)

r2_inclusivo_fe = 1 - (rss_fe / tss_fe)

print("R² inclusivo recalculado:", r2_inclusivo_fe)

# Monta a tabela-resumo
tabela_ajuste = pd.DataFrame({
    "Pooled OLS": [
        res_pooled.nobs,
        res_pooled.rsquared,
        getattr(res_pooled, "rsquared_within", np.nan),
        getattr(res_pooled, "rsquared_between", np.nan),
        getattr(res_pooled, "rsquared_overall", np.nan),
        np.nan
    ],
    "Efeitos Fixos": [
        res_fe.nobs,
        res_fe.rsquared,
        getattr(res_fe, "rsquared_within", np.nan),
        getattr(res_fe, "rsquared_between", np.nan),
        getattr(res_fe, "rsquared_overall", np.nan),
        r2_inclusivo_fe
    ],
    "Efeitos Aleatórios": [
        res_re.nobs,
        res_re.rsquared,
        getattr(res_re, "rsquared_within", np.nan),
        getattr(res_re, "rsquared_between", np.nan),
        getattr(res_re, "rsquared_overall", np.nan),
        np.nan
    ]
}, index=[
    "Observações",
    "R² principal",
    "R² within",
    "R² between",
    "R² overall",
    "R² inclusivo"
])

tabela_ajuste = tabela_ajuste.round(4)

tabela_ajuste

In [54]:
# ============================================================
# EXPORTAÇÃO DA TABELA-RESUMO DO AJUSTE DOS MODELOS
# ============================================================

tabela_ajuste.to_excel(PASTA_SAIDA + "Tabela_ajuste_modelos.xlsx")

print("Tabela de ajuste dos modelos exportada com sucesso.")

Tabela de ajuste dos modelos exportada com sucesso.


In [55]:
import numpy as np
# ============================================================
# ETAPA 23 — TABELA-RESUMO DOS DIAGNÓSTICOS
# ============================================================

tabela_diagnosticos = pd.DataFrame({
    "Teste": [
        "Hausman",
        "Wooldridge",
        "Wald modificado"
    ],
    "Hipótese nula": [
        "Efeitos Aleatórios são adequados",
        "Ausência de autocorrelação de 1ª ordem",
        "Homoscedasticidade entre municípios"
    ],
    "Estatística": [
        hausman_stat,
        F_wool,
        W_wald
    ],
    "Graus de liberdade": [
        hausman_df,
        f"{gl_num}; {gl_den}",
        gl_wald
    ],
    "p-valor": [
        hausman_p,
        p_valor_wool,
        p_valor_wald
    ],
    "Decisão": [
        "Rejeita H0; usar Efeitos Fixos" if hausman_p < 0.05 else "Não rejeita H0",
        "Rejeita H0; há autocorrelação" if p_valor_wool < 0.05 else "Não rejeita H0",
        "Rejeita H0; há heterocedasticidade" if p_valor_wald < 0.05 else "Não rejeita H0"
    ]
})

tabela_diagnosticos["Estatística"] = tabela_diagnosticos["Estatística"].astype(float).round(4)
tabela_diagnosticos["p-valor"] = tabela_diagnosticos["p-valor"].astype(float).round(4)

tabela_diagnosticos

,Teste,Hipótese nula,Estatística,Graus de liberdade,p-valor,Decisão
0,Hausman,Efeitos Aleatórios são adequados,0.0000,5,0.0,Rejeita H0; usar Efeitos Fixos
1,Wooldridge,Ausência de autocorrelação de 1ª ordem,22159.5896,1; 5569,0.0,Rejeita H0; há autocorrelação
2,Wald modificado,Homoscedasticidade entre municípios,500345.3935,5570,0.0,Rejeita H0; há heterocedasticidade


## Etapa 19 — Teste de Hausman

Nesta etapa, é realizado o teste de Hausman para comparar os modelos de Efeitos Fixos e Efeitos Aleatórios.

Quando o p-valor é menor que 0,05, rejeita-se a hipótese de adequação dos Efeitos Aleatórios, indicando que o modelo de Efeitos Fixos é o estimador mais consistente.

### Interpretação do teste de Hausman

Se o p-valor for menor que 0,05, o modelo de Efeitos Fixos é preferível ao modelo de Efeitos Aleatórios. Isso ocorre porque há evidência de correlação entre os efeitos individuais dos municípios e os regressores.

In [ ]:
# ============================================================
# ETAPA 19 — TESTE DE HAUSMAN
# ============================================================

def teste_hausman(fe, re):
    b_fe = fe.params
    b_re = re.params

    # Mantém apenas coeficientes comuns aos dois modelos
    comuns = b_fe.index.intersection(b_re.index)

    diff = b_fe[comuns] - b_re[comuns]
    cov_diff = fe.cov.loc[comuns, comuns] - re.cov.loc[comuns, comuns]

    estatistica = float(diff.T @ np.linalg.pinv(cov_diff) @ diff)
    gl = len(comuns)
    p_valor = 1 - stats.chi2.cdf(estatistica, gl)

    return estatistica, gl, p_valor

hausman_chi2, hausman_gl, hausman_p = teste_hausman(res_fe, res_re)

print("=== TESTE DE HAUSMAN ===")
print("H0: modelo de Efeitos Aleatórios é adequado")
print(f"χ² = {hausman_chi2:.2f}")
print(f"gl = {hausman_gl}")
print(f"p-valor = {hausman_p:.4f}")

if hausman_p < 0.05:
    print("Decisão: rejeita-se H0. O modelo de Efeitos Fixos é preferível.")
else:
    print("Decisão: não se rejeita H0. O modelo de Efeitos Aleatórios pode ser utilizado.")

=== TESTE DE HAUSMAN ===
H0: modelo de Efeitos Aleatórios é adequado
χ² = 520.30
gl = 5
p-valor = 0.0000
Decisão: rejeita-se H0. O modelo de Efeitos Fixos é preferível.


## Etapa 20 — Teste de Wooldridge para autocorrelação

Nesta etapa, é realizado o teste de Wooldridge para verificar autocorrelação de primeira ordem nos resíduos do modelo em painel.

A hipótese nula do teste é de ausência de autocorrelação. Assim, quando o p-valor é menor que 0,05, rejeita-se a hipótese nula, indicando que há autocorrelação nos resíduos.

Esse diagnóstico é importante porque, em dados em painel, observações do mesmo município ao longo do tempo podem estar correlacionadas. A presença de autocorrelação justifica o uso de erros-padrão robustos clusterizados por município.

O teste de Wooldridge rejeitou a hipótese nula de ausência de autocorrelação de primeira ordem, indicando dependência serial nos resíduos do painel. Esse resultado justifica o uso de erros-padrão robustos clusterizados por município.

In [ ]:
# ============================================================
# ETAPA 20 — TESTE DE WOOLDRIDGE PARA AUTOCORRELAÇÃO
# ============================================================

df_wool = (
    painel_modelo[["CO_MUNICIPIO", "ANO", "IRD"] + PREDITORES]
    .dropna()
    .copy()
    .sort_values(["CO_MUNICIPIO", "ANO"])
)

# Calcula primeiras diferenças por município
for var in ["IRD"] + PREDITORES:
    df_wool["D_" + var] = df_wool.groupby("CO_MUNICIPIO")[var].diff()

variaveis_dif = ["D_IRD"] + ["D_" + var for var in PREDITORES]

df_wool_diff = df_wool.dropna(subset=variaveis_dif).copy()

# Modelo em primeiras diferenças
X_diff = sm.add_constant(df_wool_diff[["D_" + var for var in PREDITORES]])
y_diff = df_wool_diff["D_IRD"]

modelo_fd = sm.OLS(y_diff, X_diff).fit()

# Resíduos e resíduo defasado
df_wool_diff["residuo"] = modelo_fd.resid

df_wool_diff["residuo_lag"] = (
    df_wool_diff
    .groupby("CO_MUNICIPIO")["residuo"]
    .shift(1)
)

df_wool_teste = df_wool_diff.dropna(subset=["residuo_lag"]).copy()

modelo_wool = sm.OLS(
    df_wool_teste["residuo"],
    sm.add_constant(df_wool_teste["residuo_lag"])
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_wool_teste["CO_MUNICIPIO"]}
)

coef_lag = modelo_wool.params["residuo_lag"]
erro_lag = modelo_wool.bse["residuo_lag"]

# No teste de Wooldridge, H0 implica coeficiente = -0,5
F_wool = ((coef_lag - (-0.5)) / erro_lag) ** 2

gl_num = 1
gl_den = df_wool_teste["CO_MUNICIPIO"].nunique() - 1

p_wool = 1 - stats.f.cdf(F_wool, gl_num, gl_den)

print("=== TESTE DE WOOLDRIDGE ===")
print("H0: ausência de autocorrelação de primeira ordem")
print(f"Coeficiente do resíduo defasado = {coef_lag:.4f}")
print(f"F = {F_wool:.2f}")
print(f"p-valor = {p_wool:.4f}")

if p_wool < 0.05:
    print("Decisão: rejeita-se H0. Há evidência de autocorrelação.")
else:
    print("Decisão: não se rejeita H0. Não há evidência estatística de autocorrelação.")

=== TESTE DE WOOLDRIDGE ===
H0: ausência de autocorrelação de primeira ordem
Coeficiente do resíduo defasado = 0.1709
F = 22159.59
p-valor = 0.0000
Decisão: rejeita-se H0. Há evidência de autocorrelação.


## Etapa 21 — Teste de Wald modificado para heterocedasticidade por grupo

Nesta etapa, é realizado o teste de Wald modificado para verificar heterocedasticidade entre municípios.

A hipótese nula do teste é de homocedasticidade, isto é, variância constante dos resíduos entre os grupos. Quando o p-valor é menor que 0,05, rejeita-se a hipótese nula, indicando que a variância dos resíduos difere entre os municípios.

Esse diagnóstico reforça a necessidade de utilizar erros-padrão robustos clusterizados por município, pois os erros-padrão convencionais poderiam gerar inferências inadequadas.

In [ ]:
# ============================================================
# ETAPA 21 — TESTE DE WALD MODIFICADO PARA HETEROCEDASTICIDADE
# ============================================================

residuos = res_fe.resids.to_frame(name="residuo").reset_index()
residuos.columns = ["CO_MUNICIPIO", "ANO", "residuo"]

grupo = residuos.groupby("CO_MUNICIPIO")["residuo"]

# Variância dos resíduos por município
sig2_i = grupo.apply(lambda s: (s ** 2).mean())

# Número de observações por município
T_i = grupo.size()

# Variância média geral
sig2_geral = (residuos["residuo"] ** 2).mean()

# Componente de variância do teste
V_i = grupo.apply(
    lambda s: ((s ** 2 - (s ** 2).mean()) ** 2).sum()
) / (T_i * (T_i - 1))

# Tratamento de valores inválidos
V_i = V_i.replace([np.inf, -np.inf], np.nan).dropna()
sig2_i_validos = sig2_i.loc[V_i.index]

# Estatística de Wald modificada
W_wald = (((sig2_i_validos - sig2_geral) ** 2) / V_i).sum()

gl_wald = len(sig2_i_validos)
p_wald = 1 - stats.chi2.cdf(W_wald, gl_wald)

print("=== TESTE DE WALD MODIFICADO ===")
print("H0: homocedasticidade entre municípios")
print(f"W = {W_wald:,.1f}")
print(f"gl = {gl_wald}")
print(f"p-valor = {p_wald:.4f}")

if p_wald < 0.05:
    print("Decisão: rejeita-se H0. Há evidência de heterocedasticidade entre municípios.")
else:
    print("Decisão: não se rejeita H0. Não há evidência estatística de heterocedasticidade.")

=== TESTE DE WALD MODIFICADO ===
H0: homocedasticidade entre municípios
W = 500,345.4
gl = 5570
p-valor = 0.0000
Decisão: rejeita-se H0. Há evidência de heterocedasticidade entre municípios.


O teste de Wald modificado rejeitou a hipótese nula de homocedasticidade entre municípios, indicando que a variância dos resíduos difere entre os grupos. Esse resultado, em conjunto com o teste de Wooldridge, sustenta o uso de erros-padrão robustos clusterizados por município.

## Etapa 21.1 — Síntese dos diagnósticos finais

Nesta etapa, são reunidos os principais indicadores de ajuste e diagnóstico do modelo principal.

A tabela apresenta o R² inclusivo, o R² within, o resultado do teste de Wooldridge para autocorrelação e o teste de Wald modificado para heterocedasticidade por grupo.

Essa síntese facilita a conferência dos resultados reportados no artigo e justifica o uso de erros-padrão robustos clusterizados por município.

In [ ]:
# ============================================================
# ETAPA 21.1 — SÍNTESE DOS DIAGNÓSTICOS FINAIS
# ============================================================

tabela_diagnosticos_final = pd.DataFrame({
    "Teste/Indicador": [
        "R² inclusivo — modelo principal",
        "R² within — modelo principal",
        "Teste de Wooldridge — coef. resíduo defasado",
        "Teste de Wooldridge — F",
        "Teste de Wooldridge — p-valor",
        "Teste de Wald modificado — estatística W",
        "Teste de Wald modificado — gl",
        "Teste de Wald modificado — p-valor"
    ],
    "Resultado": [
        f"{r2_inclusivo_fe:.4f}",
        f"{res_fe.rsquared:.4f}",
        f"{coef_lag:.4f}",
        f"{F_wool:.2f}",
        f"{p_wool:.4f}",
        f"{W_wald:,.1f}",
        f"{gl_wald}",
        f"{p_wald:.4f}"
    ]
})

tabela_diagnosticos_final

,Teste/Indicador,Resultado
0,R² inclusivo — modelo principal,0.7041
1,R² within — modelo principal,0.0112
2,Teste de Wooldridge — coef. resíduo defasado,0.1709
3,Teste de Wooldridge — F,22159.59
4,Teste de Wooldridge — p-valor,0.0000
5,Teste de Wald modificado — estatística W,"500,345.4"
6,Teste de Wald modificado — gl,5570
7,Teste de Wald modificado — p-valor,0.0000


## Etapa 22 — Tabela comparativa dos modelos

Nesta etapa, os resultados dos modelos Pooled OLS, Efeitos Fixos e Efeitos Aleatórios são organizados em uma tabela comparativa.

A tabela permite verificar coeficientes, estatísticas t e p-valores de cada especificação.

In [ ]:
# ============================================================
# ETAPA 22 — TABELA COMPARATIVA DOS MODELOS
# ============================================================

# Função para inserir estrelas de significância
def estrelas(p):
    if p < 0.01:
        return "***"
    elif p < 0.05:
        return "**"
    elif p < 0.10:
        return "†"
    else:
        return ""

# Função para formatar coeficiente e estatística t
def formatar_coef_t(res, var):
    if var in res.params.index:
        coef = res.params[var]
        t = res.tstats[var]
        p = res.pvalues[var]
        return f"{coef:.4f}{estrelas(p)}\n({t:.2f})"
    else:
        return ""

# Ordem das variáveis na tabela
ordem_variaveis = ["ICG", "AFD", "IED", "ATU", "const"]

# Nomes que aparecerão na tabela
nomes_linhas = {
    "ICG": "ICG",
    "AFD": "AFD",
    "IED": "IED",
    "ATU": "ATU",
    "const": "Constante"
}

# Construção da tabela principal
tabela_modelos = pd.DataFrame(index=[nomes_linhas[v] for v in ordem_variaveis])

tabela_modelos["Pooled OLS"] = [
    formatar_coef_t(res_pooled, v) for v in ordem_variaveis
]

tabela_modelos["Efeitos Fixos"] = [
    formatar_coef_t(res_fe, v) for v in ordem_variaveis
]

tabela_modelos["Efeitos Aleatórios"] = [
    formatar_coef_t(res_re, v) for v in ordem_variaveis
]

# Linhas adicionais de ajuste
tabela_modelos.loc["R² intragrupo"] = [
    f"{getattr(res_pooled, 'rsquared_within', res_pooled.rsquared):.4f}",
    f"{getattr(res_fe, 'rsquared_within', res_fe.rsquared):.4f}",
    f"{getattr(res_re, 'rsquared_within', res_re.rsquared):.4f}"
]

tabela_modelos.loc["N"] = [
    f"{res_pooled.nobs:,}".replace(",", "."),
    f"{res_fe.nobs:,}".replace(",", "."),
    f"{res_re.nobs:,}".replace(",", ".")
]

# Exibe a tabela
tabela_modelos


,Pooled OLS,Efeitos Fixos,Efeitos Aleatórios
ICG,0.1149***\n(14.68),0.0687***\n(9.78),0.0565***\n(9.40)
AFD,0.0018***\n(7.90),0.0005†\n(1.92),0.0033***\n(17.65)
IED,0.0029***\n(10.17),0.0015***\n(6.32),0.0015***\n(7.28)
ATU,-0.0139***\n(-12.08),-0.0085***\n(-6.48),-0.0129***\n(-12.49)
Constante,2.8480***\n(84.28),2.9823***\n(85.31),2.9453***\n(102.33)
R² intragrupo,0.0106,0.0089,0.0266
N,66.840,66.840,66.840


In [ ]:
# ============================================================
# EXPORTAÇÃO DA SÍNTESE DOS DIAGNÓSTICOS FINAIS
# ============================================================

tabela_diagnosticos_final.to_excel(
    os.path.join(PASTA_SAIDA, "Tabela_diagnosticos_finais.xlsx"),
    index=False
)

print("Tabela de diagnósticos finais exportada com sucesso.")

Tabela de diagnósticos finais exportada com sucesso.


## Etapa 23 — Modelo principal e robustez com controle de porte

Nesta etapa, o modelo principal de Efeitos Fixos é comparado com uma especificação alternativa que inclui o controle pelo porte da rede municipal.

O porte é representado pelo logaritmo do número de escolas públicas do município (`LN_ESC`). O objetivo é verificar se os coeficientes principais permanecem estáveis após controlar diferenças no tamanho das redes municipais.

In [51]:
# ============================================================
# ETAPA 23 — MODELO PRINCIPAL E ROBUSTEZ COM CONTROLE DE PORTE
# ============================================================

# Garante que a variável LN_ESC exista
if "LN_ESC" not in painel_modelo.columns:
    if "N_ESCOLAS" in painel_modelo.columns:
        painel_modelo["LN_ESC"] = np.log(painel_modelo["N_ESCOLAS"])
        print("Variável LN_ESC criada a partir de N_ESCOLAS.")
    else:
        raise ValueError("A base não possui N_ESCOLAS nem LN_ESC.")

# ------------------------------------------------------------
# Modelo principal — Efeitos Fixos sem controle de porte
# ------------------------------------------------------------

df_principal = (
    painel_modelo[["CO_MUNICIPIO", "ANO", "IRD"] + PREDITORES]
    .dropna()
    .copy()
    .set_index(["CO_MUNICIPIO", "ANO"])
    .sort_index()
)

y_principal = df_principal["IRD"]
X_principal = sm.add_constant(df_principal[PREDITORES])

modelo_principal = PanelOLS(
    y_principal,
    X_principal,
    entity_effects=True,
    time_effects=True
)

res_principal = modelo_principal.fit(
    cov_type="clustered",
    cluster_entity=True
)

# ------------------------------------------------------------
# Robustez com controle de porte
# ------------------------------------------------------------

df_porte = (
    painel_modelo[["CO_MUNICIPIO", "ANO", "IRD"] + PREDITORES + ["LN_ESC"]]
    .dropna()
    .copy()
    .set_index(["CO_MUNICIPIO", "ANO"])
    .sort_index()
)

y_porte = df_porte["IRD"]
X_porte = sm.add_constant(df_porte[PREDITORES + ["LN_ESC"]])

modelo_porte = PanelOLS(
    y_porte,
    X_porte,
    entity_effects=True,
    time_effects=True
)

res_porte = modelo_porte.fit(
    cov_type="clustered",
    cluster_entity=True
)

print("Modelo principal e robustez com controle de porte estimados com sucesso.")

Modelo principal e robustez com controle de porte estimados com sucesso.


In [52]:
# ============================================================
# TABELA 24 — MODELO PRINCIPAL E ROBUSTEZ COM CONTROLE DE PORTE
# ============================================================

# Função para inserir estrelas de significância
def estrelas(p):
    if p < 0.01:
        return "***"
    elif p < 0.05:
        return "**"
    elif p < 0.10:
        return "†"
    else:
        return ""

# Função para formatar coeficiente e erro-padrão
def formatar_coef_ep(res, var):
    if var in res.params.index:
        coef = res.params[var]
        ep = res.std_errors[var]
        p = res.pvalues[var]
        return f"{coef:.4f}{estrelas(p)}\n({ep:.4f})"
    else:
        return ""

# Criação da tabela no formato do artigo
tabela_modelos = pd.DataFrame({
    "Variável": [
        "Complexidade da gestão escolar — ICG",
        "Adequação da formação docente — AFD",
        "Esforço docente — IED",
        "Média de alunos por turma — ATU",
        "Porte do município — ln nº de escolas",
        "Constante",
        "Observações",
        "R² within",
        "Efeitos fixos de município",
        "Efeitos fixos de ano",
        "Erros-padrão clusterizados por município"
    ],

    "Modelo principal": [
        formatar_coef_ep(res_principal, "ICG"),
        formatar_coef_ep(res_principal, "AFD"),
        formatar_coef_ep(res_principal, "IED"),
        formatar_coef_ep(res_principal, "ATU"),
        "",
        formatar_coef_ep(res_principal, "const"),
        f"{res_principal.nobs:,.0f}".replace(",", "."),
        f"{res_principal.rsquared:.4f}",
        "Sim",
        "Sim",
        "Sim"
    ],

    "Robustez com porte": [
        formatar_coef_ep(res_porte, "ICG"),
        formatar_coef_ep(res_porte, "AFD"),
        formatar_coef_ep(res_porte, "IED"),
        formatar_coef_ep(res_porte, "ATU"),
        formatar_coef_ep(res_porte, "LN_ESC"),
        formatar_coef_ep(res_porte, "const"),
        f"{res_porte.nobs:,.0f}".replace(",", "."),
        f"{res_porte.rsquared:.4f}",
        "Sim",
        "Sim",
        "Sim"
    ]
})

tabela_modelos

,Variável,Modelo principal,Robustez com porte
0,Complexidade da gestão escolar — ICG,0.0687***\n(0.0070),0.0553***\n(0.0071)
1,Adequação da formação docente — AFD,0.0005†\n(0.0002),0.0002\n(0.0002)
2,Esforço docente — IED,0.0015***\n(0.0002),0.0016***\n(0.0002)
3,Média de alunos por turma — ATU,-0.0085***\n(0.0013),-0.0109***\n(0.0013)
4,Porte do município — ln nº de escolas,,-0.1142***\n(0.0209)
5,Constante,2.9823***\n(0.0350),3.3683***\n(0.0731)
6,Observações,66.840,66.840
7,R² within,0.0112,0.0138
8,Efeitos fixos de município,Sim,Sim
9,Efeitos fixos de ano,Sim,Sim


Nota: erros-padrão entre parênteses; erros-padrão robustos clusterizados por município. *** p < 0,01; ** p < 0,05; † p < 0,10.

## Etapa 24 — Função auxiliar para análises de robustez

Nesta etapa, é criada uma função para estimar modelos de Efeitos Fixos com efeitos de município e de ano.

Essa função será usada para padronizar as análises de robustez e reduzir repetição de código.

In [53]:
# ============================================================
# ETAPA 24 — FUNÇÃO PARA MODELOS DE EFEITOS FIXOS
# ============================================================

def rodar_efeitos_fixos(df, y_var="IRD", x_vars=None, nome="Modelo"):
    if x_vars is None:
        x_vars = PREDITORES

    df_temp = df.dropna(subset=[y_var] + x_vars).copy()
    df_temp = df_temp.set_index(["CO_MUNICIPIO", "ANO"]).sort_index()

    y_temp = df_temp[y_var]
    X_temp = sm.add_constant(df_temp[x_vars])

    modelo = PanelOLS(
        y_temp,
        X_temp,
        entity_effects=True,
        time_effects=True
    )

    resultado = modelo.fit(cov_type="clustered", cluster_entity=True)

    print("\n" + "="*80)
    print(nome)
    print("="*80)
    print(resultado.summary)

    return resultado, df_temp

## Etapa 25 — Robustez 1: exclusão dos anos de 2023 e 2024

Nesta etapa, o modelo é reestimado excluindo os anos de 2023 e 2024.

Essa análise verifica se os resultados são sensíveis à alteração metodológica introduzida pelo Inep a partir de 2023 no cálculo do IRD.

In [ ]:
# ============================================================
# ETAPA 25 — ROBUSTEZ 1: SEM 2023 E 2024
# ============================================================

df_rob1 = painel_modelo[painel_modelo["ANO"] <= 2022].copy()

res_rob1, df_rob1_panel = rodar_efeitos_fixos(
    df_rob1,
    x_vars=PREDITORES,
    nome="Robustez 1 — Sem 2023 e 2024"
)


Robustez 1 — Sem 2023 e 2024
                          PanelOLS Estimation Summary                           
Dep. Variable:                    IRD   R-squared:                        0.0092
Estimator:                   PanelOLS   R-squared (Between):              0.0366
No. Observations:               55700   R-squared (Within):               0.0038
Date:                Sun, Jul 05 2026   R-squared (Overall):              0.0270
Time:                        03:38:43   Log-likelihood                    3557.7
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      116.09
Entities:                        5570   P-value                           0.0000
Avg Obs:                      10.0000   Distribution:                 F(4,50117)
Min Obs:                      10.0000                                           
Max Obs:                      10.0000   F-statistic (robust):             34.96

## Etapa 26 — Robustez 2: inclusão da hora-aula diária

Nesta etapa, o modelo é reestimado com a inclusão da variável HAD.

Como a HAD não possui cobertura completa para todo o período, ela não entra no modelo principal e é utilizada apenas como teste de robustez.

In [ ]:
# ============================================================
# ETAPA 26 — ROBUSTEZ 2: INCLUSÃO DA HAD
# ============================================================

if "HAD" in painel_modelo.columns:
    df_rob2 = painel_modelo[painel_modelo["ANO"].between(2016, 2024)].copy()
    pred_rob2 = PREDITORES + ["HAD"]

    res_rob2, df_rob2_panel = rodar_efeitos_fixos(
        df_rob2,
        x_vars=pred_rob2,
        nome="Robustez 2 — Inclusão da HAD"
    )
else:
    res_rob2 = None
    print("A variável HAD não foi encontrada na base.")


Robustez 2 — Inclusão da HAD
                          PanelOLS Estimation Summary                           
Dep. Variable:                    IRD   R-squared:                        0.0145
Estimator:                   PanelOLS   R-squared (Between):              0.0417
No. Observations:               50130   R-squared (Within):               0.0182
Date:                Sun, Jul 05 2026   R-squared (Overall):              0.0352
Time:                        03:38:47   Log-likelihood                    5411.5
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      130.77
Entities:                        5570   P-value                           0.0000
Avg Obs:                       9.0000   Distribution:                 F(5,44547)
Min Obs:                       9.0000                                           
Max Obs:                       9.0000   F-statistic (robust):             39.06

## Etapa 27 — Robustez 3: estimação com anos espaçados

Nesta etapa, o modelo é reestimado utilizando anos espaçados.

Essa estratégia reduz a sobreposição temporal da janela móvel utilizada no cálculo do IRD e verifica se os resultados permanecem estáveis.

In [ ]:
# ============================================================
# ETAPA 27 — ROBUSTEZ 3: ANOS ESPAÇADOS
# ============================================================

ANOS_ESPACADOS = [2013, 2016, 2019, 2022]

df_rob3 = painel_modelo[painel_modelo["ANO"].isin(ANOS_ESPACADOS)].copy()

res_rob3, df_rob3_panel = rodar_efeitos_fixos(
    df_rob3,
    x_vars=PREDITORES,
    nome="Robustez 3 — Anos espaçados"
)


Robustez 3 — Anos espaçados
                          PanelOLS Estimation Summary                           
Dep. Variable:                    IRD   R-squared:                        0.0087
Estimator:                   PanelOLS   R-squared (Between):              0.0389
No. Observations:               22280   R-squared (Within):               0.0067
Date:                Sun, Jul 05 2026   R-squared (Overall):              0.0290
Time:                        03:38:52   Log-likelihood                    426.33
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      36.655
Entities:                        5570   P-value                           0.0000
Avg Obs:                       4.0000   Distribution:                 F(4,16703)
Min Obs:                       4.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             20.732

## Etapa 28 — Robustez 4: controle pelo porte da rede municipal

Nesta etapa, o modelo é reestimado com controle pelo porte da rede municipal.

O controle é feito por meio do logaritmo do número de escolas públicas do município. Essa análise verifica se os resultados se mantêm mesmo após considerar o tamanho da rede.

In [ ]:
# ============================================================
# ETAPA 28 — ROBUSTEZ 4: CONTROLE PELO PORTE DA REDE
# ============================================================

# Primeiro, verifica possíveis nomes para a variável de número de escolas
possiveis_colunas_porte = [
    "N_ESCOLAS", "QT_ESCOLAS", "QTD_ESCOLAS", "NUM_ESCOLAS",
    "N_ESCOLAS_PUBLICAS", "QT_ESCOLAS_PUBLICAS", "qtd_escolas_publicas",
    "num_escolas_publicas"
]

coluna_porte = None

for col in possiveis_colunas_porte:
    if col in painel_modelo.columns:
        coluna_porte = col
        break

if coluna_porte is not None:
    df_rob4 = painel_modelo.copy()
    df_rob4["ln_escolas"] = np.log(df_rob4[coluna_porte])

    pred_rob4 = PREDITORES + ["ln_escolas"]

    res_rob4, df_rob4_panel = rodar_efeitos_fixos(
        df_rob4,
        x_vars=pred_rob4,
        nome=f"Robustez 4 — Controle de porte ({coluna_porte})"
    )
else:
    res_rob4 = None
    print("Nenhuma coluna de número de escolas foi encontrada.")
    print("Colunas disponíveis na base:")
    print(list(painel_modelo.columns))


Robustez 4 — Controle de porte (N_ESCOLAS)
                          PanelOLS Estimation Summary                           
Dep. Variable:                    IRD   R-squared:                        0.0138
Estimator:                   PanelOLS   R-squared (Between):              0.0590
No. Observations:               66840   R-squared (Within):               0.0143
Date:                Sun, Jul 05 2026   R-squared (Overall):              0.0444
Time:                        03:38:56   Log-likelihood                    582.10
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      171.97
Entities:                        5570   P-value                           0.0000
Avg Obs:                       12.000   Distribution:                 F(5,61254)
Min Obs:                       12.000                                           
Max Obs:                       12.000   F-statistic (robust):    

## Etapa 29 — Síntese das análises de robustez

Nesta etapa, os resultados do modelo principal e das especificações alternativas são reunidos em uma tabela.

A finalidade é verificar se os sinais e a significância dos coeficientes permanecem estáveis diante de diferentes decisões metodológicas.

In [ ]:
# ============================================================
# ETAPA 29 — SÍNTESE DAS ANÁLISES DE ROBUSTEZ
# ============================================================

def resumo_coeficientes(res, nome):
    if res is None:
        return None

    tabela = pd.DataFrame({
        nome: res.params,
        f"p_{nome}": res.pvalues
    })

    return tabela

lista_resumos = [
    resumo_coeficientes(res_fe, "Principal"),
    resumo_coeficientes(res_rob1, "Sem_2023_2024"),
    resumo_coeficientes(res_rob2, "Com_HAD"),
    resumo_coeficientes(res_rob3, "Anos_espacados"),
    resumo_coeficientes(res_rob4, "Controle_porte")
]

lista_resumos = [tab for tab in lista_resumos if tab is not None]

tabela_robustez = lista_resumos[0]

for tab in lista_resumos[1:]:
    tabela_robustez = tabela_robustez.join(tab, how="outer")

tabela_robustez = tabela_robustez.round(4)

tabela_robustez

,Principal,p_Principal,Sem_2023_2024,p_Sem_2023_2024,Com_HAD,p_Com_HAD,Anos_espacados,p_Anos_espacados,Controle_porte,p_Controle_porte
AFD,0.0005,0.0545,0.0002,0.4486,0.0004,0.0859,0.0004,0.2675,0.0002,0.4508
ATU,-0.0085,0.0000,-0.0068,0.0000,-0.0123,0.0000,-0.0060,0.0010,-0.0109,0.0000
HAD,NaN,NaN,NaN,NaN,0.0082,0.0375,NaN,NaN,NaN,NaN
ICG,0.0687,0.0000,0.0659,0.0000,0.0702,0.0000,0.0653,0.0000,0.0553,0.0000
IED,0.0015,0.0000,0.0014,0.0000,0.0014,0.0000,0.0016,0.0000,0.0016,0.0000
const,2.9823,0.0000,2.9575,0.0000,3.0293,0.0000,2.9301,0.0000,3.3683,0.0000
ln_escolas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.1142,0.0000


In [ ]:
# Exportação da tabela de robustez

tabela_robustez.to_excel(PASTA_SAIDA + "Tabela_6_robustez.xlsx")

print("Tabela de robustez exportada.")

Tabela de robustez exportada.


## Etapa 30 — Exportação geral dos resultados

Nesta etapa, as principais tabelas geradas no notebook são exportadas para um único arquivo Excel.

Essa exportação facilita a conferência dos resultados e a utilização das tabelas no artigo.

In [ ]:
# ============================================================
# ETAPA 30 — EXPORTAÇÃO GERAL DOS RESULTADOS
# ============================================================

arquivo_saida = PASTA_SAIDA + "resultados_artigo_ird.xlsx"

with pd.ExcelWriter(arquivo_saida) as writer:
    desc.to_excel(writer, sheet_name="Descritivas")
    corr.to_excel(writer, sheet_name="Correlacao")
    vif_transversal.to_excel(writer, sheet_name="VIF_transversal", index=False)
    vif_within.to_excel(writer, sheet_name="VIF_intragrupo", index=False)
    tabela_modelos.to_excel(writer, sheet_name="Modelos")
    tabela_robustez.to_excel(writer, sheet_name="Robustez")

print("Arquivo exportado com sucesso:")
print(arquivo_saida)

Arquivo exportado com sucesso:
/content/saidas/resultados_artigo_ird.xlsx


## Etapa 31 — Conferência final da reprodutibilidade

Nesta etapa, são apresentadas as principais informações finais do notebook.

Essa conferência permite verificar se o notebook reproduz corretamente a base e os modelos descritos no artigo.

In [ ]:
# ============================================================
# ETAPA 31 — CONFERÊNCIA FINAL DA REPRODUTIBILIDADE
# ============================================================

print("Notebook executado com sucesso.")
print("Observações do modelo principal:", painel_modelo.shape[0])
print("Municípios:", painel_modelo["CO_MUNICIPIO"].nunique())
print("Período:", painel_modelo["ANO"].min(), "-", painel_modelo["ANO"].max())
print("Variável dependente: IRD")
print("Preditores:", PREDITORES)
print("Pasta de resultados:", PASTA_SAIDA)

Notebook executado com sucesso.
Observações do modelo principal: 66840
Municípios: 5570
Período: 2013 - 2024
Variável dependente: IRD
Preditores: ['ICG', 'AFD', 'IED', 'ATU']
Pasta de resultados: /content/saidas/


# Encerramento

Este notebook reproduz as principais etapas empíricas do artigo: leitura e validação da base, estatísticas descritivas, matriz de correlação, diagnóstico de multicolinearidade, estimação dos modelos em painel, teste de Hausman e análises de robustez.

A estrutura foi organizada para permitir a conferência sequencial dos procedimentos adotados e garantir a reprodutibilidade dos resultados apresentados no estudo.

In [50]:
# ============================================================
# CONFERÊNCIA FINAL DAS SAÍDAS
# ============================================================

import os
import pandas as pd

print("Pasta de saída:", PASTA_SAIDA)
print("\nArquivos gerados:")

for arquivo in os.listdir(PASTA_SAIDA):
    caminho = os.path.join(PASTA_SAIDA, arquivo)
    tamanho_mb = os.path.getsize(caminho) / (1024 ** 2)
    print(f"- {arquivo} ({tamanho_mb:.2f} MB)")

Pasta de saída: /content/saidas/

Arquivos gerados:
- Tabela_2_estatisticas_descritivas.xlsx (0.00 MB)
- Tabela_diagnosticos_finais.xlsx (0.00 MB)
- Tabela_VIF.xlsx (0.01 MB)
- Tabela_4_matriz_correlacao.xlsx (0.00 MB)
- resultados_artigo_ird.xlsx (0.01 MB)
- Tabela_ajuste_modelos.xlsx (0.00 MB)
